In [4]:
import os
import dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.memory import ChatMessageHistory,ConversationSummaryBufferMemory

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm=ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=40
)

In [6]:
chat_history = ChatMessageHistory()
memory=ConversationSummaryBufferMemory(
    llm=llm,
    chat_memory=chat_history,
    max_token_limit=40,
    return_messages=True,
    memory_key='history'
)

memory.save_context(
    inputs={"input": "你好，我叫小明"},  # 用户输入（键固定为"input"）
    outputs={"output": "小明你好！有什么能帮你的？"}  # AI输出（键固定为"output"）
)

memory.save_context(
    inputs={"input": "帮我推荐一本Python入门书"},
    outputs={"output": "推荐《Python编程：从入门到实践》，适合零基础入门。"}
)

#读取记忆（短消息保留，超限时生成总结）
memory_result = memory.load_memory_variables({})
print("记忆中的对话内容：")
for msg in memory_result["history"]:
    print(f"{msg.type.upper()}: {msg.content}")

记忆中的对话内容：
SYSTEM: The human introduces himself as Xiao Ming. The AI responds by greeting Xiao Ming and asking how it can assist him.
HUMAN: 帮我推荐一本Python入门书
AI: 推荐《Python编程：从入门到实践》，适合零基础入门。


模拟客服交互

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains.llm import LLMChain

# 1、初始化大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=500
)

# 2、定义提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


# 3、创建带摘要缓冲的记忆系统
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=400,
    memory_key="chat_history",
    return_messages=True,
)

# 4、创建对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
    verbose=False
)

# 5、模拟多轮对话
dialogue = [
    ("你好，我想查询订单12345的状态", None),
    ("这个订单是上周五下的", None),
    ("我现在急着用，能加急处理吗", None),
    ("等等，我可能记错订单号了，应该是12346", None),
    ("对了，你们退货政策是怎样的", None)
]

# 6、执行对话
for user_input, _ in dialogue:
    response = chain.invoke({"input": user_input})
    print(f"用户: {user_input}")
    print(f"客服: {response['text']}\n")

# 7、查看当前记忆状态
print("\n=== 当前记忆内容 ===")
print(memory.load_memory_variables({}))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29468\2891928696.py:28: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


用户: 你好，我想查询订单12345的状态
客服: 你好！感谢你联系我。关于订单12345的状态，我会帮你查询一下。请稍等片刻。 

（假如有具体的查询结果，可以在这里提供，例如：）  
你的订单12345目前的状态是“已发货”，预计在3个工作日内送达。如果你还有其他问题，欢迎随时问我！

用户: 这个订单是上周五下的
客服: 谢谢你提供的信息！根据你在上周五下的订单12345，目前的状态是“已发货”，预计在3个工作日内送达。如果你有任何其他问题或者需要进一步的帮助，请随时告诉我！

用户: 我现在急着用，能加急处理吗
客服: 我理解你的着急心情！关于加急处理订单的问题，我会尽力帮你。请问你能提供一下你的订单号和相关信息吗？我会向相关部门反馈，看看是否可以尽快处理你的订单。谢谢你的耐心！

用户: 等等，我可能记错订单号了，应该是12346
客服: 没问题！谢谢你更正订单号。让我来查询一下订单12346的状态。请稍等片刻。

（假如有具体的查询结果，可以在这里提供，例如：）  
你的订单12346目前的状态是“已发货”，预计在3个工作日内送达。我会尽量帮你联系相关部门，看看是否可以加急处理。如果有其他问题，请随时告诉我！

用户: 对了，你们退货政策是怎样的
客服: 我们的退货政策如下：

1. **退货期限**：一般情况下，您可以在收到商品后的30天内申请退货。
2. **退货条件**：商品必须保持未使用、未拆封的状态，并且保留原包装及配件。
3. **申请流程**：您可以通过我们的客服中心提交退货申请，填写相关信息后，我们会为您提供退货地址和进一步的指导。
4. **退款方式**：一旦我们收到退回的商品并确认符合退货条件，退款将会在3-5个工作日内处理，并原路返回到您的支付账户。

如果您有具体的商品需要退货，或者还有其他问题，请随时告诉我，我会很乐意帮助您！


=== 当前记忆内容 ===
{'chat_history': [SystemMessage(content="The human inquires about the status of order 12345. The AI thanks the human for reaching out and offers to check the status. After a brief pause, 